# 04 — Fit the three-class head, and report honestly

**Gate this notebook closes: G4 — the project's own bar.**

Two minutes of compute and the whole argument of the project. What is being fitted is a
`Linear(512, 3)` on cached features; what is being *decided* is whether this pipeline can
tell invasive carcinoma from carcinoma still inside a duct.

Three rules, each of which has sunk a pathology model before:

- **Class weights**, inversely proportional to frequency. Without them the model answers
  `non_epithelium` to everything, scores ~80 % accuracy, and never finds a tumour. This
  notebook *demonstrates* that rather than asserting it.
- **Grouped splits.** `GroupKFold` by slide for model selection, then one report on the
  **held-out institutions** `OL LL E2 EW GM S3`. Never a random tile split.
- **The invasive ↔ non-invasive cell reported on its own.** Never folded into a macro
  average. That one number is the project.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import backend_path            # noqa: F401 - puts the demo backend on sys.path
backend_path.ensure_dirs()
print("backend :", backend_path.BACKEND_ROOT)
print("tiles   :", backend_path.TILES_DIR)
print("models  :", backend_path.MODELS_DIR)

In [ ]:
import json

import numpy as np
import torch

import bcss, datasets, models, report

INIT = "imagenet"
INVERT_POLARITY = False
EPOCHS = 400          # full-batch on cached features. Seconds.
LR = 1e-3
WEIGHT_DECAY = 1e-4
SEED = 0

torch.manual_seed(SEED)

rows = datasets.read_manifest(backend_path.TILES_DIR / "tiles_manifest.csv")
cache = np.load(backend_path.FEATURES_DIR / f"{INIT}{'_inverted' if INVERT_POLARITY else ''}.npz")

# The check that prevents the quietest failure in the plan: features paired with the
# wrong manifest train to about chance and look like a model that will not learn.
assert str(cache["fingerprint"]) == datasets.manifest_fingerprint(rows), (
    "the feature cache was built from a different manifest. Re-run notebook 03."
)

features = cache["features"]
labels = cache["labels"]
print(f"{len(rows):,} tiles, {features.shape[1]}-d features, init={INIT}")

## The head, and the fit

A linear head on frozen features is a convex problem up to the optimiser, so there is no
early-stopping ritual to perform: a fixed number of full-batch steps at a fixed learning rate
is reproducible and takes seconds.

In [ ]:
# The fit lives in src/train.py, imported rather than redefined here.
#
# Both approaches must fit *identically* - approach 3 differs in exactly one respect,
# where the body's weights came from - so if each notebook ran its own copy of the
# optimiser loop, the A/B would be free to become a comparison of two learning rates
# instead. One definition, three callers: this notebook, approach 3's, and the scripts
# that actually produced the published checkpoints.
import train

print("recipe:", {"epochs": train.EPOCHS, "lr": train.LR,
                  "weight_decay": train.WEIGHT_DECAY, "seed": train.SEED})

fit_head, predict = train.fit_head, train.predict

## Model selection: GroupKFold by slide, on the training institutions only

The held-out institutions are not touched here. Tiles from one slide are near-duplicates —
same patient, same scanner, same stain batch, often literal cells at a shared edge — so the
folds are grouped by slide and the leak is asserted rather than assumed.

In [ ]:
index_of = {row["tile_id"]: i for i, row in enumerate(rows)}
train_rows, test_rows = datasets.institution_split(rows)
datasets.assert_no_leak(train_rows, test_rows)

train_idx = np.array([index_of[r["tile_id"]] for r in train_rows])
test_idx = np.array([index_of[r["tile_id"]] for r in test_rows])
print(f"train {len(train_idx):,} tiles / {len({r['slide_id'] for r in train_rows})} slides")
print(f"test  {len(test_idx):,} tiles / {len({r['slide_id'] for r in test_rows})} slides"
      f"  ({sorted({r['institution'] for r in test_rows})})")

weights = datasets.class_weights(train_rows)
print(f"\nclass weights: {dict(zip(bcss.CLASS_NAMES, [round(float(w), 3) for w in weights]))}")

folds = datasets.slide_folds(train_rows, folds=5, seed=SEED)
fold_scores = []
for k, held in enumerate(folds):
    fit_rows = [train_rows[i] for i in range(len(train_rows)) if i not in set(held.tolist())]
    val_rows = [train_rows[i] for i in held]
    datasets.assert_no_leak(fit_rows, val_rows)

    fit_i = np.array([index_of[r["tile_id"]] for r in fit_rows])
    val_i = np.array([index_of[r["tile_id"]] for r in val_rows])

    head, _ = fit_head(features[fit_i], labels[fit_i], weights=weights)
    scored = report.evaluate(labels[val_i], predict(head, features[val_i]))
    fold_scores.append(scored.dice_invasive)
    print(f"  fold {k}: dice invasive vs in-situ = {scored.dice_invasive:.3f} "
          f"({len(val_rows):,} tiles)")

print(f"\nGroupKFold: {np.mean(fold_scores):.3f} +/- {np.std(fold_scores):.3f}")
print("The spread is the number to keep in mind when approach 3 claims +3 points.")

## The report: held-out institutions

One fit on all the training institutions, one evaluation on the six held out. This is the
number that leaves the notebook.

In [ ]:
head, final_loss = fit_head(features[train_idx], labels[train_idx], weights=weights)
predicted = predict(head, features[test_idx])

# Tumour content is the *region's* invasive share, read off the labels in notebook 02 -
# a property of the tissue, never of a prediction.
summary = json.loads((backend_path.TILES_DIR / "export_summary.json").read_text())
share_of = {r["roi_id"]: r["invasive_share"] for r in summary["per_region"]}

scored = report.evaluate(
    labels[test_idx],
    predicted,
    slide_ids=np.array([r["slide_id"] for r in test_rows]),
    tumour_content=np.array([share_of[r["roi_id"]] for r in test_rows]),
)

print(report.format_matrix(scored.matrix))
print()
print(scored.headline())
print(f"\naccuracy {scored.accuracy:.3f} | macro F1 {scored.macro_f1:.3f}")
for name, value in scored.per_class_dice.items():
    print(f"  dice {name:<26} {value:.3f}")
print(f"\nrecall: invasive {scored.invasive_recall:.3f} | in-situ {scored.non_invasive_recall:.3f}")

### Stratified by tumour content

A slide that is 80–90 % tumour is easy. A 2–3 mm invasive focus inside a 2–3 cm block, mixed
with DCIS and normal ducts, is the hard case *and the clinically common one*. A model that
averages well can be useless on exactly the cases that matter, so every bin carries its tile
count — a bin holding forty tiles is a curiosity, not a result.

In [ ]:
print(f"{'tumour content':<16}{'tiles':>9}{'accuracy':>11}{'dice inv':>11}")
for row in scored.by_tumour_content:
    accuracy = "-" if row["accuracy"] is None else f"{row['accuracy']:.3f}"
    dice = "-" if row["dice_invasive"] is None else f"{row['dice_invasive']:.3f}"
    print(f"{row['bin']:<16}{row['tiles']:>9,}{accuracy:>11}{dice:>11}")

### The class-weight ablation

Not a formality. The unweighted model is the one a careless run produces, and it looks
respectable on accuracy while never finding a tumour. Showing it is the evidence behind the
claim.

In [ ]:
unweighted, _ = fit_head(features[train_idx], labels[train_idx],
                         weights=torch.ones(len(bcss.CLASS_NAMES)))
naive = report.evaluate(labels[test_idx], predict(unweighted, features[test_idx]))

print("without class weights")
print(report.format_matrix(naive.matrix))
print(f"  accuracy {naive.accuracy:.3f}  <- respectable")
print(f"  {naive.headline()}")
print()
print(f"with class weights:    accuracy {scored.accuracy:.3f}, "
      f"dice invasive {scored.dice_invasive:.3f}")
print(f"without:               accuracy {naive.accuracy:.3f}, "
      f"dice invasive {naive.dice_invasive:.3f}")
print("\nIf the unweighted accuracy is the higher of the two, that is the point being made:")
print("accuracy is not the objective, and the confusion matrix is why.")

In [ ]:
# Keep everything notebook 06 needs to publish, and everything approach 3 needs to
# compare against - including the per-slide scores, which are what the paired test uses.
result = {
    "init": INIT,
    "invert_polarity": INVERT_POLARITY,
    "final_loss": final_loss,
    "groupkfold_dice": {"mean": float(np.mean(fold_scores)), "sd": float(np.std(fold_scores)),
                        "folds": [float(s) for s in fold_scores]},
    "held_out": {
        "institutions": sorted({r["institution"] for r in test_rows}),
        "tiles": int(scored.tiles),
        "accuracy": scored.accuracy,
        "macro_f1": scored.macro_f1,
        "dice_invasive_vs_non_invasive": scored.dice_invasive,
        "per_class_dice": scored.per_class_dice,
        "confusion": scored.matrix.tolist(),
        "invasive_called_non_invasive": scored.invasive_called_non_invasive,
        "non_invasive_called_invasive": scored.non_invasive_called_invasive,
        "by_tumour_content": scored.by_tumour_content,
        "per_slide_dice": scored.by_slide,
    },
    "ablation_no_class_weights": {
        "accuracy": naive.accuracy,
        "dice_invasive_vs_non_invasive": naive.dice_invasive,
        "confusion": naive.matrix.tolist(),
    },
}
path = backend_path.REPORTS_DIR / f"04_report_{INIT}{'_inverted' if INVERT_POLARITY else ''}.json"
path.write_text(json.dumps(result, indent=2), encoding="utf-8")
torch.save(head.state_dict(), backend_path.REPORTS_DIR / f"04_head_{INIT}.pt")
print(f"wrote {path.name} and the fitted head")

TARGET = 0.75
print(f"\ndice invasive vs non-invasive: {scored.dice_invasive:.3f}  (target >= {TARGET})")
if scored.dice_invasive >= TARGET:
    print("G4 OK - go to notebook 06 and pin it.")
elif scored.dice_invasive >= 0.60:
    print("Short of target. Notebook 05 (fine-tune layer4) is the next lever.")
else:
    print("Well short. Do NOT reach for a bigger model - at this level the usual cause")
    print("is a label or alignment fault. Re-read gate G2's eyeball panel first.")

---

- [ ] G4a — confusion matrix on **held-out institutions**, not a random split
- [ ] G4b — invasive ↔ non-invasive Dice reported alone, target ≥ 0.75
- [ ] G4c — tumour-content table printed, with tile counts
- [ ] G4d — class-weight ablation run and recorded

**Then run approach 3's notebooks** — the SimCLR initialisation is one download and one
feature pass, and it is the cheapest thing on the ablation list that might move this number.